# 实验3 GPU复现入口

本次已实测硬件为本机RTX 3050；本notebook是T4/其他GPU上的重建与复测入口，未宣称已在Kaggle T4运行。先启用GPU，并将完整交付ZIP作为自己的数据输入；安装与该环境CUDA/Python相容的PyTorch和TensorRT。原机生成的engine须在新GPU上重新构建。

In [ ]:
import torch, platform
print(platform.python_version(), torch.__version__, torch.version.cuda)
assert torch.cuda.is_available(), "请先启用GPU"
print(torch.cuda.get_device_name())
import tensorrt as trt
print("TensorRT", trt.__version__)
assert int(trt.__version__.split(".")[0]) >= 10


In [ ]:
from pathlib import Path
import zipfile, subprocess, sys
archives=list(Path("/kaggle/input").rglob("实验3_优化完整代码模型数据与报告.zip"))
assert len(archives)==1, archives
root=Path("/kaggle/working/exp3");root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z: z.extractall(root)
p=root/"优化";g=p/"gpu"
def run(name,*args):
    subprocess.run([sys.executable,str(g/"scripts"/name),*args],cwd=g,check=True)


## 在当前GPU重建原A2引擎并复测

软件包版本需要与当前GPU运行环境匹配。下面所有硬件名称、计时和精度都由实际执行写入JSON，不用本机RTX3050数字代替。

In [ ]:
run("build_trt.py")
run("bench_exp3.py","--model","models/a2_original.pt","--device","cuda")
run("bench_pipeline.py","--model","models/a2_trt_fp16.engine","--device","cuda")
run("eval_detail.py","--model","models/a2_original.pt")
run("eval_pipeline.py","--model","models/a2_trt_fp16.engine")


## 可选：重新执行蒸馏与温度筛选

运行于此处解压后的副本。完整训练使用12轮；检查点按验证集选出。请先完成训练，再测速。

In [ ]:
# run("distill_yolo.py","--epochs","12","--microbatch","16","--fresh")
# run("temperature_study.py")
# run("finish_temperature.py")


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/exp3_gpu_results","zip",g/"results")
print("当前GPU结果已保存：/kaggle/working/exp3_gpu_results.zip")
